# Week 07 · NLP Take-Home Assignment
## TF-IDF, Sentiment, & Embeddings
This notebook contains solutions to Q1 and Q2 using the generated ShopSense Dataset.

In [1]:
import pandas as pd
import numpy as np
from collections import Counter
import math
from sklearn.feature_extraction.text import TfidfVectorizer

# Load dataset
df = pd.read_csv('../data/ShopSense_Reviews.csv')
print(f"Dataset shape: {df.shape}")
df.head()


Dataset shape: (10000, 6)


,review_id,customer_id,product_id,category,review_text,rating
0,0,2424,704,Books,good okay okay very very item the,4
1,1,1106,877,Electronics,earbuds battery sound charging screen screen e...,2
2,2,7201,180,Books,the bad item the good good product a product v...,5
3,3,6977,266,Food,a very good okay okay good okay good okay a the,3
4,4,5422,755,Food,very very very good a is okay is the,5


## Q1 (a)
Compute the full TF-IDF matrix for all 10,000 reviews from scratch.

In [2]:
# Tokenizer
def tokenize(text):
    if not isinstance(text, str):
        return []
    return text.lower().split()

docs = df['review_text'].tolist()
# Calculate term frequencies
tf_list = []
df_counts = Counter()
for doc in docs:
    tokens = tokenize(doc)
    counts = Counter(tokens)
    tf_list.append(counts)
    for term in set(tokens):
        df_counts[term] += 1

N = len(docs)
idf = {}
for term, df_count in df_counts.items():
    # standard IDF formulation: ln((1 + N)/(1 + df_count)) + 1
    # We will use sklearn's default formula to match Q1(c) later
    idf[term] = math.log((1 + N) / (1 + df_count)) + 1

# Sorting vocabulary to maintain indices
vocab = sorted(list(idf.keys()))
vocab_index = {word: i for i, word in enumerate(vocab)}

print(f"Vocabulary size: {len(vocab)}")


Vocabulary size: 31


We have successfully computed the required stats. Let's create the sparse matrix for TF-IDF.

In [3]:
from scipy.sparse import lil_matrix, csr_matrix

# Build TF-IDF matrix
tfidf_matrix = lil_matrix((N, len(vocab)), dtype=np.float32)

for i, tf_dict in enumerate(tf_list):
    doc_len = sum(tf_dict.values())
    if doc_len == 0:
        continue
    # compute Euclidean norm for L2 normalization
    row_norm = 0.0
    for term, count in tf_dict.items():
        if term in vocab_index:
            tfidf_val = count * idf[term]
            row_norm += tfidf_val ** 2
    row_norm = math.sqrt(row_norm) if row_norm > 0 else 1.0
    
    for term, count in tf_dict.items():
        if term in vocab_index:
            j = vocab_index[term]
            tfidf_val = count * idf[term]
            tfidf_matrix[i, j] = tfidf_val / row_norm  # standard L2 norm

tfidf_matrix = tfidf_matrix.tocsr()
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")


TF-IDF matrix shape: (10000, 31)


## Q1 (b)
Given the query 'wireless earbuds battery life poor', rank the top-5 most relevant reviews using cosine similarity against TF-IDF vectors.

In [4]:
from scipy.sparse.linalg import norm as spnorm

query = 'wireless earbuds battery life poor'
query_tokens = tokenize(query)
# create query vector
query_tf = Counter(query_tokens)
query_vec = np.zeros(len(vocab))
for term, count in query_tf.items():
    if term in vocab_index:
        query_vec[vocab_index[term]] = count * idf[term]

# normalize query
q_norm = np.linalg.norm(query_vec)
if q_norm > 0:
    query_vec = query_vec / q_norm

# compute cosine sim (dot product since both are L2 normalized)
cos_sim = tfidf_matrix.dot(query_vec)
top_5_indices = np.argsort(cos_sim)[::-1][:5]

print("Top 5 Results:")
for idx in top_5_indices:
    print(f"Review {idx} (Score: {cos_sim[idx]:.4f}): {df.iloc[idx]['review_text']}")


Top 5 Results:
Review 3016 (Score: 0.9409): earbuds poor life wireless battery poor wireless the
Review 4368 (Score: 0.9308): earbuds life wireless very battery battery poor the
Review 2268 (Score: 0.9304): earbuds life wireless the poor is battery very the
Review 391 (Score: 0.9299): earbuds earbuds is life poor battery battery wireless the
Review 2548 (Score: 0.9137): earbuds the battery earbuds excellent wireless life life the wireless battery life poor the


## Q1 (c)
Compare your output against sklearn's TfidfVectorizer. Report the average L2 difference between the two matrices.

In [5]:
from scipy.sparse.linalg import norm as spnsnorm
vectorizer = TfidfVectorizer(lowercase=True, token_pattern=r'(?u)\b\w+\b')
sklearn_tfidf = vectorizer.fit_transform(docs)

# Align vocabularies just in case, but TfidfVectorizer standard parses out short tokens differently.
# To make a direct math compare, let's just use the exact tokenizer for sklearn:
vectorizer_custom = TfidfVectorizer(tokenizer=tokenize, token_pattern=None, lowercase=False)
sklearn_custom_tfidf = vectorizer_custom.fit_transform(docs)

diff = tfidf_matrix - sklearn_custom_tfidf
avg_l2_diff = spnsnorm(diff) / N
print(f"Average L2 difference between custom and sklearn TF-IDF: {avg_l2_diff:.8f}")


Average L2 difference between custom and sklearn TF-IDF: 0.00000000


## Q1 (d)
Identify the single word with the highest average TF-IDF score across the 'Electronics' category. Explain why that word ranks at the top.

In [6]:
electronics_idx = df[df['category'] == 'Electronics'].index
electronics_matrix = tfidf_matrix[electronics_idx]
avg_tfidf_electronics = np.array(electronics_matrix.mean(axis=0))[0]

max_idx = np.argmax(avg_tfidf_electronics)
top_word = vocab[max_idx]

print(f"Top word in Electronics: {top_word}")
print(f"Average TF-IDF in Electronics: {avg_tfidf_electronics[max_idx]:.4f}")


Top word in Electronics: earbuds
Average TF-IDF in Electronics: 0.3360


### Reason:
The word 'earbuds' ranks at the top because it has a high frequency within the 'Electronics' category document subset (high TF) but is relatively rare across the entire dataset (e.g. absent in 'Clothing', 'Food', etc.), meaning it has a high IDF. High TF * High IDF = Top Score.

## Q2 (a)
Compute TF('fabric', Doc_42), IDF('fabric', 10K corpus), and TF-IDF('fabric', Doc_42). Show every arithmetic step.

In [7]:
# 10K corpus stats
target_doc_id = 42
text_42 = df.loc[target_doc_id, 'review_text']
print(f"Doc_42: {text_42}")

tokens_42 = tokenize(text_42)
tf_fabric = tokens_42.count('fabric')

df_fabric = df_counts.get('fabric', 0)
print(f"Frequency of 'fabric' in Doc_42 (TF): {tf_fabric}")
print(f"Document Frequency of 'fabric' in corpus (DF): {df_fabric}")
print(f"Total documents (N): {N}")

idf_fabric = math.log((1 + N) / (1 + df_fabric)) + 1
print(f"IDF('fabric') = ln((1 + {N}) / (1 + {df_fabric})) + 1 = {idf_fabric:.4f}")

tfidf_raw_fabric = tf_fabric * idf_fabric
print(f"Raw TF-IDF('fabric', Doc_42) = {tf_fabric} * {idf_fabric:.4f} = {tfidf_raw_fabric:.4f}")

# L2 Norm representation
doc_42_len = sum((tokens_42.count(w) * idf[w])**2 for w in set(tokens_42)) ** 0.5
tfidf_norm_fabric = tfidf_raw_fabric / doc_42_len
print(f"L2 Normalized TF-IDF('fabric', Doc_42) = {tfidf_norm_fabric:.4f}")


Doc_42: the fabric of this is great and the embroidery is nice
Frequency of 'fabric' in Doc_42 (TF): 1
Document Frequency of 'fabric' in corpus (DF): 875
Total documents (N): 10000
IDF('fabric') = ln((1 + 10000) / (1 + 875)) + 1 = 3.4351
Raw TF-IDF('fabric', Doc_42) = 1 * 3.4351 = 3.4351
L2 Normalized TF-IDF('fabric', Doc_42) = 0.1564


## Q2 (b)
Compute IDF('the') and IDF('embroidery'). Explain in 2 sentences why IDF('the') approaches 0 while IDF('embroidery') is high.

In [8]:
for w in ['the', 'embroidery']:
    df_w = df_counts.get(w, 0)
    idf_w = math.log((1 + N) / (1 + df_w)) + 1
    print(f"DF({w}) = {df_w}, IDF({w}) = {idf_w:.4f}")


DF(the) = 10000, IDF(the) = 1.0000
DF(embroidery) = 883, IDF(embroidery) = 3.4260


**Explanation:**
'the' is a stop word present in practically every document (DF approaches N), making its IDF computation `ln(1) + 1` which is close to 1.0 (or 0 if not smoothed), severely reducing its weight. Conversely, 'embroidery' only occasionally appears in clothing documents (low DF), resulting in a much larger fraction `N/DF` inside the logarithm, pushing its IDF score extremely high.

## Q2 (c)
Write a 3-sentence rebuttal to: 'Why not just use word frequency? TF-IDF is overcomplicated.'

**Rebuttal:**
Simply using word frequency over-rewards common stop words like 'the' or 'is' which carry little semantic meaning. TF-IDF elegantly discounts these ubiquitous terms by multiplying frequency with Inverse Document Frequency (IDF), highlighting terms that are frequent in a specific document but rare globally. This dual-balancing mechanism ensures that domain-specific keywords accurately represent document uniqueness instead of grammatical filler.

## Q2 (Bonus)
Re-run using BM25 weighting (k1=1.5, b=0.75) for the same query. How do scores change?

In [9]:
# BM25 Implementation
k1 = 1.5
b = 0.75
avgdl = sum(len(tokenize(d)) for d in docs) / N

def compute_bm25(query_tokens, doc_idx, doc_tokens):
    score = 0.0
    dl = len(doc_tokens)
    for q in query_tokens:
        if q not in df_counts: continue
        tf = doc_tokens.count(q)
        # BM25 IDF: ln((N - DF + 0.5) / (DF + 0.5) + 1)
        idf_bm25 = math.log((N - df_counts[q] + 0.5) / (df_counts[q] + 0.5) + 1)
        term1 = tf * (k1 + 1)
        term2 = tf + k1 * (1 - b + b * (dl / avgdl))
        score += idf_bm25 * (term1 / term2)
    return score

bm25_scores = []
for i, d in enumerate(docs):
    bm25_scores.append(compute_bm25(query_tokens, i, tokenize(d)))

bm25_scores = np.array(bm25_scores)
top_5_bm25_idx = np.argsort(bm25_scores)[::-1][:5]

print("Top 5 Results using BM25:")
for idx in top_5_bm25_idx:
    print(f"Review {idx} (Score: {bm25_scores[idx]:.4f}): {df.iloc[idx]['review_text']}")


Top 5 Results using BM25:
Review 3016 (Score: 14.7840): earbuds poor life wireless battery poor wireless the
Review 2548 (Score: 14.2173): earbuds the battery earbuds excellent wireless life life the wireless battery life poor the
Review 391 (Score: 13.9699): earbuds earbuds is life poor battery battery wireless the
Review 4508 (Score: 13.9699): earbuds battery battery poor wireless sound life earbuds the
Review 4368 (Score: 13.7345): earbuds life wireless very battery battery poor the


**How scores change:**
BM25 ranks documents slightly differently than cosine TF-IDF because it incorporates document length normalization explicitly `(dl / avgdl)` differently from L2 norm, and term frequency saturation limits the influence of term spamming (due to the `k1` factor). Also the underlying IDF computation differs slightly.